# Create KidneyCure Awards (American Society of Nephrology / ASN Foundation for Kidney Research)

Creates awards from the recipient lists that **KidneyCure** (the ASN Foundation
for Kidney Research, the American Society of Nephrology's research foundation,
US 501(c)3 since 2012; the older programmes were ASN's own) publishes at
https://www.kidneycure.org/pages/recipients.aspx — one static HTML page per
programme, grouped by award year (and, on the Lipps page, by named
sub-fellowship):

- Transition to Independence family: Carl W. Gottschalk Research Scholar Grant,
  John Merrill Grant in Transplantation, Norman Siegel Research Scholar Grant,
  Joseph V. Bonventre Career Development Grant, Norman and Sandra Coplon
  Research Scholar Grant, KidneyCure Research Scholar Grant, AAIM Junior
  Development Grant in Geriatric Nephrology, Alaska Kidney Foundation, Halpin
  Foundation, NephCure and OHF partner grants, William and Sandra Bennett
  Clinical Scholars Program
- Fellowships: Ben J. Lipps Research Fellowship Program (incl. named
  fellowships), Pre-Doctoral Fellowship Award Program

**437 awards, 1996-2026** (local run 2026-10-01). Title 91.5% (pre-~2004
entries list the name only), lead name 100%. The lists publish **no grant
number, amount, institution or exact dates**: amount/currency are NULL (the
§6.7 amount check is waived — the funder does not publish amounts per award),
`start_date` is 1 January of the listed award year (year-only precedent: EFSD,
Naito, Arthritis Foundation), `end_date` NULL.

**Scope:** every programme on the recipients hub is kept, including the
fellowships (training pathway, in scope per the 2026-10-01 rule) and the
Bennett Clinical Scholars (education-research grants; kept and flagged).

**Prerequisites:**
- Run `scripts/local/kidneycure_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/kidneycure/kidneycure_projects.parquet` (§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** synthetic and stable,
`KIDNEYCURE-{programme code}-{year}-{name slug}` (e.g.
`KIDNEYCURE-GOT-2020-mark-r-hanudel`). KidneyCure prints no grant number, and the
125 existing citation stubs on F4320306605 (crossref_work_funders,
crossref_work.grants, europepmc_work_funders) carry co-funders' numbers (NIH
R01/K23/T32, AHA SDG) or free text ("2014 Gottschalk"), so 0 stubs collapse.

**Funder:** shipped under the tracker's F-id **F4320306605 American Society of
Nephrology** (US, ror 014pcm848, doi 10.13039/100001463). Twin: **F4320311111
ASN Foundation for Kidney Research** (same ROR, doi 10.13039/100009257, 14 stub
rows, 0 collapse) is the legal awarding body since 2012; flagged for the
coordinator, not mapped.

**Priority:** `567` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.kidneycure_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/kidneycure/kidneycure_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_awards FROM openalex.awards.kidneycure_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.kidneycure_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.kidneycure_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320306605 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320306605;

## Step 2: Create KidneyCure Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.kidneycure_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320306605  -- American Society of Nephrology
),

g AS (
    SELECT
        *,
        TRY_CAST(award_year AS INT) AS yr,
        NULLIF(TRIM(title), '') AS title_clean,
        NULLIF(TRIM(lead_family_name), '') AS family,
        NULLIF(TRIM(lead_given_name), '') AS given
    FROM openalex.awards.kidneycure_raw
    WHERE funder_award_id IS NOT NULL AND TRIM(funder_award_id) != ''
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    -- Project title; name-only (pre-~2004) listings get "Programme Year: Name".
    COALESCE(g.title_clean,
             CONCAT(TRIM(g.programme), ' ', g.award_year, ': ', TRIM(g.lead_name))) as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    g.funding_type as funding_type,
    -- Named sub-fellowship where the page gives one (Lipps page <h6>), else the programme.
    COALESCE(NULLIF(TRIM(g.sub_programme), ''), TRIM(g.programme)) as funder_scheme,
    'kidneycure_recipients' as provenance,
    CASE WHEN g.yr IS NOT NULL THEN TRY_TO_DATE(CONCAT(g.award_year, '-01-01'), 'yyyy-MM-dd') END as start_date,
    CAST(NULL AS DATE) as end_date,
    g.yr as start_year,
    CAST(NULL AS INT) as end_year,
    CASE WHEN g.family IS NOT NULL THEN named_struct(
        'given_name', g.given,
        'family_name', g.family,
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', CAST(NULL AS STRING),
            'country', CAST(NULL AS STRING),
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'kidneycure_recipients' AND priority = 567;

-- Insert into openalex_awards_raw with priority.
-- Priority 567: direct-from-funder ingest of KidneyCure's own recipient lists.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    567 as priority
FROM openalex.awards.kidneycure_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids, COUNT(DISTINCT funder_award_id) as distinct_award_ids
FROM openalex.awards.kidneycure_awards;

In [ ]:
%sql
-- §2.1.1 shape check: every funder_award_id is the synthetic KIDNEYCURE-{CODE}-{YEAR}-{slug} form.
SELECT
    COUNT(*) as total,
    SUM(CASE WHEN funder_award_id RLIKE '^KIDNEYCURE-[A-Z]+-(19|20)[0-9]{2}-[a-z0-9-]+$' THEN 1 ELSE 0 END) as well_formed
FROM openalex.awards.kidneycure_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, start_year,
       lead_investigator.given_name, lead_investigator.family_name, landing_page_url
FROM openalex.awards.kidneycure_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, funding_type, COUNT(*) as cnt, MIN(start_year) as first_year, MAX(start_year) as last_year
FROM openalex.awards.kidneycure_awards
GROUP BY funder_scheme, funding_type ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.kidneycure_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.kidneycure_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.kidneycure_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Coverage (expect ~91% project titles, 100% lead names and start years; no amounts published).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name
FROM openalex.awards.kidneycure_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'kidneycure_recipients'
GROUP BY provenance, priority;